# Build a 3-sigma cost-anomaly detector that pages you before finance does

Your AWS bill went up $1,200 last month and the team only noticed when finance asked. The fix is not a budget alert in the console (those fire too late and too coarse). The fix is a detector that reads the Cost and Usage Report, computes a rolling baseline per service, and pages on-call when the latest day breaks the band. You build it once, it watches forever, and the next $400 spike has a name on it inside 24 hours.

Real CUR has a 24-hour delivery lag and a 30-column Resource Groups Tagging schema. Waiting a full day mid-lab is not a viable forcing function for analytical work, so this lab seeds 30 days of synthetic CUR-shaped data with a deliberate anomaly on day 25 and asks you to build the detector that catches it on day 26. The reflection covers what changes when you point this same code at a real CUR delivery.

Four deliverables map to four checkpoints:
1. Seeded CUR table is queryable; baseline cost-per-service shape verifies the anomaly is sitting where the brief says it is.
2. Anomaly detection query computes 3-sigma over a 14-day rolling window per service and flags exactly day 25 for `Amazon Athena`.
3. Lambda detector reads the query result and publishes to SNS only when at least one anomaly row exists. Silent on clean replays.
4. EventBridge daily schedule is wired to invoke the detector once a day in a specific hour window.

**Time.** About 75 minutes hands-on. The Lambda packaging and the EventBridge target wiring are the long poles; the SQL itself is shorter than it looks.

**Cost.** The detector itself costs nothing. The lab spends Athena pennies running the anomaly query a half-dozen times during checkpoints (under 100 MB scanned across the session, somewhere between $0.20 and $0.80 depending on debugging re-runs). Lambda invokes and SNS publishes at lab volume are inside the always-free tier. S3 storage of the seed dataset is under 5 MB and rounds to nothing. The cleanup cell at the bottom deletes every resource so the bill stops the moment you finish.

This lab maps to Data Reliability Engineering: cost as a first-class observability signal. The same pattern, swapped to real CUR, is what FinOps-aware teams ship to production.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import csv
import getpass
import io
import json
import random
import time
import zipfile
from datetime import date, datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "cost-anomaly-detection"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"  # all track labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
BUCKET_NAME = None  # resolved after STS
DB = f"labrun-{LAB_ID}-db"
TABLE = "cur_synthetic"
WG_NAME = f"labrun-{LAB_ID}-wg"
DETECTOR_ROLE = f"labrun-{LAB_ID}-detector-role"
DETECTOR_INLINE_POLICY = f"labrun-{LAB_ID}-detector-inline"
DETECTOR_NAME = f"labrun-{LAB_ID}-detector"
TOPIC_NAME = f"labrun-{LAB_ID}-alerts"
SCHEDULE_NAME = f"labrun-{LAB_ID}-daily"
TOPIC_ARN = None  # resolved after sns.create_topic

# S3 prefixes inside the bucket.
CUR_PREFIX = "cur/"
ATHENA_RESULTS_PREFIX = "athena-results/"
REPORT_PREFIX = "report/"
ANOMALY_QUERY_KEY = f"{REPORT_PREFIX}anomaly_query.sql"

# Seed config. 30 days, 5 services, one row per (day, service). Day 25
# (zero-indexed) has Amazon Athena's cost at 4x baseline, which is the
# anomaly the detector must catch.
SEED = 1729
NUM_DAYS = 30
ANOMALY_DAY_INDEX = 25  # zero-based; day_25 is the 26th day
SERVICES = [
    "Amazon S3",
    "Amazon Athena",
    "AWS Glue",
    "AWS Lambda",
    "Amazon SNS",
]
BASELINE = {
    "Amazon S3": 4.20,
    "Amazon Athena": 10.00,
    "AWS Glue": 6.50,
    "AWS Lambda": 0.40,
    "Amazon SNS": 0.05,
}
ANOMALY_SERVICE = "Amazon Athena"
ANOMALY_MULTIPLIER = 4.2

# Anchor date is fixed (not today) so seed values are deterministic across runs.
ANCHOR_DATE = date(2026, 1, 1)
ANOMALY_DATE = ANCHOR_DATE + timedelta(days=ANOMALY_DAY_INDEX)
CLEAN_DATE = ANCHOR_DATE + timedelta(days=ANOMALY_DAY_INDEX + 1)  # day 26, no spike

print(f"Anchor date: {ANCHOR_DATE.isoformat()} (day 0)")
print(f"Anomaly date: {ANOMALY_DATE.isoformat()} (day {ANOMALY_DAY_INDEX}, {ANOMALY_SERVICE})")
print(f"Clean replay date: {CLEAN_DATE.isoformat()} (day {ANOMALY_DAY_INDEX + 1})")

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All track-data-engineering labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve names that depend on the account ID.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
TOPIC_ARN = f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{TOPIC_NAME}"
DETECTOR_LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{DETECTOR_NAME}"
print(f"Bucket name resolved: {BUCKET_NAME}")
print(f"Topic ARN resolved: {TOPIC_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is in reverse-creation order. No critical (hourly-billed)
# resources in this lab; standard order is sufficient.
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).

CLEANUP_MANIFEST: list[CleanupResource] = [
    CleanupResource(
        type="eventbridge_rule",
        id=SCHEDULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {SCHEDULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {SCHEDULE_NAME}"
        ),
    ),
    CleanupResource(
        type="lambda_function",
        id=DETECTOR_NAME,
        region=REGION,
        cli_delete_command=f"aws lambda delete-function --function-name {DETECTOR_NAME}",
    ),
    CleanupResource(
        type="sns_topic",
        id=TOPIC_NAME,
        region=REGION,
        cli_delete_command=f"aws sns delete-topic --topic-arn {TOPIC_ARN}",
    ),
    CleanupResource(
        type="iam_role",
        id=DETECTOR_ROLE,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {DETECTOR_ROLE}",
    ),
    CleanupResource(
        type="athena_workgroup",
        id=WG_NAME,
        region=REGION,
        cli_delete_command=f"aws athena delete-work-group --work-group {WG_NAME} --recursive-delete-option",
    ),
    CleanupResource(
        type="glue_table",
        id=TABLE,
        parent=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-table --database-name {DB} --name {TABLE}",
    ),
    CleanupResource(
        type="glue_database",
        id=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DB}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    Runs run_cleanup against the manifest. Errors are swallowed because
    atexit handlers must not raise; the dedicated cleanup cell prints the
    full structured report and is the authoritative cleanup entry point.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind."""
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("remove them manually with the AWS CLI commands printed by the")
    print("cleanup cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Seed the synthetic CUR dataset and stand up an Athena table over it

Real CUR delivery lands a CSV/Parquet drop in S3 once every 24 hours, with a 30-column schema covering line item type, savings plan attribution, tags, and reservation amortization. Waiting a day for the first row in a 75-minute lab does not work, so the seed cell writes a CUR-shaped CSV with three columns (`usage_date`, `service`, `unblended_cost`) covering 30 days and 5 services. Day 25 has `Amazon Athena` running at 4x its baseline; every other day for every service stays inside its normal envelope.

Build it in this order:
1. Create the S3 bucket (`labrun-cost-anomaly-detection-{account-id}`) and tag it with `labrun:lab-slug=cost-anomaly-detection`.
2. Generate the seed CSV bytes deterministically from `SEED`. The cell below builds the bytes for you; uncomment nothing. The output is one CSV file at `s3://{bucket}/cur/cur.csv` with a header row plus 150 data rows.
3. Create the Glue database `labrun-cost-anomaly-detection-db` and an external table `cur_synthetic` over `s3://{bucket}/cur/` using `LazySimpleSerDe` with comma terminator, skip-header set to 1, and the three columns typed `string, string, double` (Athena handles the `usage_date` cast inside the anomaly query in Task 2).
4. Create the Athena workgroup `labrun-cost-anomaly-detection-wg` with engine v3, results location `s3://{bucket}/athena-results/`, and a `BytesScannedCutoffPerQuery=1000000000` data-scan cap so a runaway query cannot rack up dollars.

The checkpoint runs two Athena selects: one to confirm the row count is 150 (30 days * 5 services), and one to confirm `Amazon Athena` on the anomaly date is approximately 4x its baseline ($42 vs the $10 baseline; the seed adds small Gaussian noise so the value lands in a tight envelope around $42).

In [ ]:
# NBVAL_SKIP
# Task 1: Seed synthetic CUR, create the Glue database/table, create the
# Athena workgroup with a scan cap.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Generate the CUR-shaped CSV deterministically from SEED. Every (day, service)
# row uses baseline + small Gaussian noise. Day 25 "Amazon Athena" multiplies
# by ANOMALY_MULTIPLIER so the 3-sigma detector has something to catch.
rnd = random.Random(SEED)
buf = io.StringIO()
writer = csv.writer(buf)
writer.writerow(["usage_date", "service", "unblended_cost"])
for day_idx in range(NUM_DAYS):
    d = ANCHOR_DATE + timedelta(days=day_idx)
    for svc in SERVICES:
        base = BASELINE[svc]
        noise = rnd.gauss(0, base * 0.015)  # 1.5% sigma, well inside 3-sigma envelope
        cost = base + noise
        if day_idx == ANOMALY_DAY_INDEX and svc == ANOMALY_SERVICE:
            cost = base * ANOMALY_MULTIPLIER  # the spike, no noise
        writer.writerow([d.isoformat(), svc, f"{cost:.4f}"])
seed_bytes = buf.getvalue().encode("utf-8")
print(f"Generated {NUM_DAYS * len(SERVICES)} CUR rows ({len(seed_bytes)} bytes)")

# Create and tag the bucket. us-east-1 rejects LocationConstraint; other regions require it.
# YOUR CODE: Create the S3 bucket with s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 does NOT accept CreateBucketConfiguration.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Upload the seed CSV under cur/ so Athena can scan it.
# YOUR CODE: Upload the seed bytes to s3 using s3.put_object() with
# Bucket=BUCKET_NAME, Key=f"{CUR_PREFIX}cur.csv", and Body=seed_bytes.
print(f"Seed uploaded to s3://{BUCKET_NAME}/{CUR_PREFIX}cur.csv")

# Glue database.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DB,
            "Description": f"labrun {LAB_ID} synthetic CUR database",
        }
    )
    print(f"Created Glue database: {DB}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DB} already exists, reusing.")
    else:
        raise

# External table over s3://{bucket}/cur/. The LazySimpleSerDe with comma
# field terminator handles the CSV; skip.header.line.count=1 ignores the header.
# YOUR CODE: Create the Glue external table using glue.create_table() with
# DatabaseName=DB and TableInput shaped as below. The table must point at
# Location=f"s3://{BUCKET_NAME}/{CUR_PREFIX}" with three columns
# (usage_date string, service string, unblended_cost double), use
# org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe with
# field.delim="," and a skip.header.line.count=1 table parameter.
# Reference shape (use this as the TableInput dict):
#   {
#     "Name": TABLE,
#     "TableType": "EXTERNAL_TABLE",
#     "Parameters": {
#         "classification": "csv",
#         "skip.header.line.count": "1",
#     },
#     "StorageDescriptor": {
#         "Columns": [
#             {"Name": "usage_date", "Type": "string"},
#             {"Name": "service", "Type": "string"},
#             {"Name": "unblended_cost", "Type": "double"},
#         ],
#         "Location": f"s3://{BUCKET_NAME}/{CUR_PREFIX}",
#         "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
#         "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
#         "SerdeInfo": {
#             "SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe",
#             "Parameters": {"field.delim": ","},
#         },
#     },
#   }
# Wrap in try/except ClientError for AlreadyExistsException.
print(f"Created Glue table: {DB}.{TABLE} over s3://{BUCKET_NAME}/{CUR_PREFIX}")

# Athena workgroup with results location and a 1 GB per-query scan cap.
try:
    athena.create_work_group(
        Name=WG_NAME,
        Configuration={
            "ResultConfiguration": {
                "OutputLocation": f"s3://{BUCKET_NAME}/{ATHENA_RESULTS_PREFIX}",
            },
            "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"},
            "BytesScannedCutoffPerQuery": 1_000_000_000,
            "EnforceWorkGroupConfiguration": True,
            "PublishCloudWatchMetricsEnabled": False,
        },
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Created Athena workgroup: {WG_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("InvalidRequestException",) and "already exists" in str(e).lower():
        print(f"Athena workgroup {WG_NAME} already exists, reusing.")
    else:
        raise

print()
print("Task 1 build complete. Glue catalog points at the seeded CUR; Athena workgroup is capped at 1 GB per query.")

In [ ]:
# NBVAL_SKIP
# Helper: run an Athena query in our workgroup, poll until terminal, return rows.
# This is shared by checkpoints and the Task 2 / Task 3 cells, so it lives at
# module scope. The helper passes the workgroup name; results land in the
# workgroup's configured output location.

def run_athena_query(sql: str, timeout_s: int = 60) -> list[list[str]]:
    a = boto3.client(
        "athena",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    qid = a.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DB},
        WorkGroup=WG_NAME,
    )["QueryExecutionId"]
    print(f"Asking Athena nicely for results, query id {qid[:8]}...")
    deadline = time.time() + timeout_s
    while True:
        info = a.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = info["Status"]["State"]
        if state in ("SUCCEEDED",):
            break
        if state in ("FAILED", "CANCELLED"):
            reason = info["Status"].get("StateChangeReason", "")
            raise RuntimeError(f"Athena query {state}: {reason}")
        if time.time() > deadline:
            raise TimeoutError(f"Athena query did not finish in {timeout_s}s (state={state})")
        time.sleep(1.0)
    rows = []
    paginator = a.get_paginator("get_query_results")
    for page in paginator.paginate(QueryExecutionId=qid):
        for row in page["ResultSet"]["Rows"]:
            rows.append([col.get("VarCharValue", "") for col in row["Data"]])
    return rows


print("Athena helper registered.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: CUR table queryable, baseline shape verifies the day-25 anomaly.

def checkpoint_1(session):
    try:
        # Total row count.
        rows = run_athena_query(f'SELECT count(*) FROM "{DB}"."{TABLE}"')
        if len(rows) < 2:
            return CheckpointResult(status="fail", error_reason=f"count(*) returned no data rows (got {rows!r}).")
        total = int(rows[1][0])
        expected_total = NUM_DAYS * len(SERVICES)
        if total != expected_total:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected {expected_total} rows in {TABLE} ({NUM_DAYS} days * {len(SERVICES)} services), "
                    f"got {total}. Check the seed upload to s3://{BUCKET_NAME}/{CUR_PREFIX}."
                ),
            )

        # Anomaly-day sum for Amazon Athena.
        anomaly_date_str = ANOMALY_DATE.isoformat()
        rows = run_athena_query(
            f'SELECT service, sum(unblended_cost) FROM "{DB}"."{TABLE}" '
            f"WHERE usage_date = '{anomaly_date_str}' AND service = '{ANOMALY_SERVICE}' GROUP BY service"
        )
        data = [r for r in rows[1:] if r and r[0]]
        if not data:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No rows for service={ANOMALY_SERVICE!r} on usage_date={anomaly_date_str}. "
                    f"Verify the seed file uploaded and the table Location points at s3://{BUCKET_NAME}/{CUR_PREFIX}."
                ),
            )
        spike_value = float(data[0][1])
        baseline_value = BASELINE[ANOMALY_SERVICE]
        expected_low = baseline_value * (ANOMALY_MULTIPLIER - 0.5)
        expected_high = baseline_value * (ANOMALY_MULTIPLIER + 0.5)
        if not (expected_low <= spike_value <= expected_high):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Anomaly-day cost for {ANOMALY_SERVICE} on {anomaly_date_str} is {spike_value:.2f}; "
                    f"expected approximately {baseline_value * ANOMALY_MULTIPLIER:.2f} "
                    f"(allowed range {expected_low:.2f}-{expected_high:.2f}). The seed file may be wrong."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs two Athena selects against `cur_synthetic`. If the first one returns a non-150 count, the seed upload or the table Location is wrong. If the second one returns nothing or the wrong number, the table is reading the seed but the spike row is missing or the table Location is pointed at the wrong prefix.

</details>

<details><summary>Hint 2 (direction)</summary>

Three things have to be true: `s3.put_object` wrote the CSV to `s3://{BUCKET_NAME}/cur/cur.csv`, the Glue table's `Location` is `s3://{BUCKET_NAME}/cur/` (with the trailing slash; Athena scans every object under the prefix), and the SerDe has `field.delim=","` plus `skip.header.line.count=1`. If you forget the skip-header parameter Athena treats the header row as data and the `unblended_cost` column casts fail.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=f"{CUR_PREFIX}cur.csv",
    Body=seed_bytes,
)

try:
    glue.create_table(
        DatabaseName=DB,
        TableInput={
            "Name": TABLE,
            "TableType": "EXTERNAL_TABLE",
            "Parameters": {
                "classification": "csv",
                "skip.header.line.count": "1",
            },
            "StorageDescriptor": {
                "Columns": [
                    {"Name": "usage_date", "Type": "string"},
                    {"Name": "service", "Type": "string"},
                    {"Name": "unblended_cost", "Type": "double"},
                ],
                "Location": f"s3://{BUCKET_NAME}/{CUR_PREFIX}",
                "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
                "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
                "SerdeInfo": {
                    "SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe",
                    "Parameters": {"field.delim": ","},
                },
            },
        },
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Table {DB}.{TABLE} already exists, reusing.")
    else:
        raise
```

</details>

**Wallet check.** Athena scanned about 5 KB total across the two checkpoint queries; at $5/TB that is too small to round to a cent. S3 storage of the seed is under 5 KB, also free. Glue catalog operations (database, table) are inside the always-free 1M-objects/1M-requests tier. Workgroup creation is free. Total damage so far: too small to measure. Your coffee this morning cost about 200,000 times more.

## Task 2: Author the 3-sigma anomaly detection query and store it in S3

The detector logic is one query. The Lambda in Task 3 will read the SQL text from S3 (so you can edit the query without re-deploying the Lambda), execute it against `cur_synthetic`, and page if any rows return. The exam-relevant work is shaping the SQL correctly.

A 3-sigma detector over a 14-day rolling window per service has two pieces:
1. A CTE that, for every row, computes the mean and standard deviation of the prior 14 days for that service. Use `AVG(unblended_cost) OVER (...)` and `STDDEV_SAMP(unblended_cost) OVER (...)` with a `PARTITION BY service ORDER BY usage_date ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING` window. The `1 PRECEDING` boundary excludes the current row so the baseline is not contaminated by the day being scored.
2. An outer select that emits one row per anomaly. The filter is `unblended_cost > mean_cost + 3 * stddev_cost`. Also filter `stddev_cost IS NOT NULL` so the first 14 days of the window (which have no prior data) do not show up.

Write the SQL, upload it to `s3://{bucket}/report/anomaly_query.sql`, and run it once locally to confirm it returns exactly one row for `Amazon Athena` on the anomaly date. The checkpoint pulls the SQL from S3 and runs it the same way the Lambda will.

In [ ]:
# NBVAL_SKIP
# Task 2: Author the anomaly query, upload to s3://.../report/anomaly_query.sql,
# run it once locally to confirm it flags day 25 Amazon Athena.

# YOUR CODE: Set ANOMALY_QUERY_SQL to a CTE-based Athena/Trino query against
# "{DB}"."{TABLE}" that:
#   - In the CTE, computes mean_cost and stddev_cost over a 14-day rolling
#     window per service (PARTITION BY service ORDER BY usage_date ROWS
#     BETWEEN 14 PRECEDING AND 1 PRECEDING) using AVG() and STDDEV_SAMP().
#   - In the outer select, returns usage_date, service, unblended_cost,
#     mean_cost, stddev_cost where stddev_cost IS NOT NULL AND
#     unblended_cost > mean_cost + 3 * stddev_cost.
# Use the f-string substitutions f'"{DB}"."{TABLE}"' so the SQL references
# the actual database and table names.

# Upload the SQL to S3 so the Lambda can read it at runtime.
# YOUR CODE: Upload ANOMALY_QUERY_SQL to s3 using s3.put_object() with
# Bucket=BUCKET_NAME, Key=ANOMALY_QUERY_KEY, and Body=ANOMALY_QUERY_SQL.encode("utf-8").
print(f"Anomaly query uploaded to s3://{BUCKET_NAME}/{ANOMALY_QUERY_KEY}")

# Run the query locally to confirm it flags day 25 Amazon Athena.
rows = run_athena_query(ANOMALY_QUERY_SQL, timeout_s=90)
data_rows = rows[1:] if rows else []
print(f"Anomaly query returned {len(data_rows)} row(s):")
for r in data_rows:
    print("  -", r)

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Stored SQL exists in S3, runs successfully against the seeded
# data, and returns exactly one row for the anomaly date and Amazon Athena.

def checkpoint_2(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=ANOMALY_QUERY_KEY)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("NoSuchKey", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Anomaly query not found at s3://{BUCKET_NAME}/{ANOMALY_QUERY_KEY}. "
                        f"Make sure Task 2 uploads the SQL string to that key."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        sql_text = obj["Body"].read().decode("utf-8")
        sql_lower = sql_text.lower()
        required_tokens = ["stddev_samp", "avg(", "rows between", "preceding", "partition by", "service"]
        missing = [t for t in required_tokens if t not in sql_lower]
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Stored SQL is missing required pieces: {missing}. "
                    f"The 3-sigma detector needs AVG + STDDEV_SAMP over a "
                    f"PARTITION BY service ROWS BETWEEN ... PRECEDING window."
                ),
            )

        rows = run_athena_query(sql_text, timeout_s=90)
        data_rows = rows[1:] if rows else []
        if len(data_rows) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Stored query returned {len(data_rows)} row(s); expected exactly 1 "
                    f"({ANOMALY_SERVICE} on {ANOMALY_DATE.isoformat()}). "
                    f"Re-check the 3*stddev threshold and the stddev_cost IS NOT NULL filter."
                ),
            )
        row = data_rows[0]
        joined = " ".join(row).lower()
        if ANOMALY_SERVICE.lower() not in joined:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Stored query returned 1 row but the service is not {ANOMALY_SERVICE!r}: {row}. "
                    f"The seeded anomaly is on {ANOMALY_SERVICE}; another service flagging means the "
                    f"window or threshold is off."
                ),
            )
        if ANOMALY_DATE.isoformat() not in joined:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Stored query returned 1 row but the usage_date is not {ANOMALY_DATE.isoformat()}: {row}. "
                    f"Verify the window excludes the current row (ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

3-sigma is a window-function exercise. Athena (Trino) supports `OVER` windows. The SQL is the right surface; you do not need to write Python loops to compute the rolling mean.

</details>

<details><summary>Hint 2 (direction)</summary>

Use `AVG(unblended_cost) OVER (...)` and `STDDEV_SAMP(unblended_cost) OVER (...)` inside a CTE. Window frame: `PARTITION BY service ORDER BY usage_date ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING`. The `1 PRECEDING` boundary excludes the current row from its own baseline. Outer select filters `stddev_cost IS NOT NULL AND unblended_cost > mean_cost + 3 * stddev_cost`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
ANOMALY_QUERY_SQL = f'''WITH stats AS (
  SELECT
    usage_date,
    service,
    unblended_cost,
    AVG(unblended_cost) OVER (
      PARTITION BY service
      ORDER BY usage_date
      ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING
    ) AS mean_cost,
    STDDEV_SAMP(unblended_cost) OVER (
      PARTITION BY service
      ORDER BY usage_date
      ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING
    ) AS stddev_cost
  FROM "{DB}"."{TABLE}"
)
SELECT
  usage_date,
  service,
  unblended_cost,
  mean_cost,
  stddev_cost
FROM stats
WHERE stddev_cost IS NOT NULL
  AND unblended_cost > mean_cost + 3 * stddev_cost
ORDER BY usage_date, service'''

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=ANOMALY_QUERY_KEY,
    Body=ANOMALY_QUERY_SQL.encode("utf-8"),
)
```

</details>

**Wallet check.** Two more Athena queries against ~5 KB each. Total Athena scan-bytes for the lab so far is still measured in kilobytes, not megabytes. S3 storage for the SQL text is a few hundred bytes. Running cost: about two cents if Athena's billing rounded up. Your coffee this morning cost 200x more.

## Task 3: Build the Lambda detector, wire it to SNS, prove it pages on anomalies and stays silent on clean replays

The Lambda is the runtime piece of the detector. EventBridge in Task 4 will invoke it once a day; you can also invoke it directly to test. The function reads the SQL from S3, runs the Athena query, parses the row count, and publishes to SNS only when at least one row returns. The exam-relevant pattern is: keep the Lambda dumb, keep the SQL in S3, keep the threshold inside the SQL.

Build it in this order:
1. Create the SNS topic `labrun-cost-anomaly-detection-alerts` and tag it. `sns.create_topic` is idempotent on the name and returns the topic ARN whether the topic is new or already existed.
2. Create the IAM role `labrun-cost-anomaly-detection-detector-role` with a trust policy that lets `lambda.amazonaws.com` assume it, attach the `AWSLambdaBasicExecutionRole` managed policy, and attach a single inline policy that grants:
   - `athena:StartQueryExecution`, `athena:GetQueryExecution`, `athena:GetQueryResults` on the workgroup.
   - `glue:GetTable`, `glue:GetDatabase`, `glue:GetPartitions` on the database and table.
   - `s3:GetObject`, `s3:ListBucket`, `s3:PutObject`, `s3:GetBucketLocation` on the bucket and bucket objects (for the seed CSV, the SQL text, and the Athena results location).
   - `sns:Publish` on the topic.
3. Build a Lambda handler in memory using the standard library `zipfile` module so the lab is fully self-contained. Runtime `python3.13`. The handler runs the anomaly query, polls until terminal, and publishes to SNS only when result rows exist beyond the header.
4. Create the Lambda function with the inline zip, set environment variables (`BUCKET`, `DB`, `TABLE`, `WG`, `TOPIC_ARN`, `QUERY_KEY`), and wait the customary IAM propagation buffer.
5. Invoke the Lambda twice from the notebook: once with `target_date` set to the anomaly date (the spike must trigger an SNS publish), once with a clean date (no publish). The checkpoint repeats this with `cloudwatch.get_metric_statistics` reading `AWS/SNS NumberOfMessagesPublished` for the topic in the last 5 minutes and compares the publish counts before and after.

In [ ]:
# NBVAL_SKIP
# Task 3: SNS topic, IAM role + inline policy, Lambda function with inline zip,
# wire and invoke.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lam = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sns = boto3.client(
    "sns",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# SNS topic. create_topic is idempotent on the name.
# YOUR CODE: Call sns.create_topic(Name=TOPIC_NAME, Tags=[...]) and save the
# returned ARN to topic_arn. Tags format:
# [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
assert topic_arn == TOPIC_ARN, f"topic_arn mismatch: {topic_arn!r} != {TOPIC_ARN!r}"
print(f"SNS topic ready: {topic_arn}")

# IAM role for the Lambda. Trust policy lets the Lambda service assume it.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
try:
    iam.create_role(
        RoleName=DETECTOR_ROLE,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description=f"labrun {LAB_ID} detector role",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Created role: {DETECTOR_ROLE}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {DETECTOR_ROLE} already exists, reusing.")
    else:
        raise

iam.attach_role_policy(
    RoleName=DETECTOR_ROLE,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "athena:StartQueryExecution",
                "athena:GetQueryExecution",
                "athena:GetQueryResults",
            ],
            "Resource": f"arn:aws:athena:{REGION}:{ACCOUNT_ID}:workgroup/{WG_NAME}",
        },
        {
            "Effect": "Allow",
            "Action": [
                "glue:GetTable",
                "glue:GetDatabase",
                "glue:GetPartitions",
            ],
            "Resource": [
                f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:catalog",
                f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DB}",
                f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:table/{DB}/{TABLE}",
            ],
        },
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:GetBucketLocation",
            ],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
        {
            "Effect": "Allow",
            "Action": "sns:Publish",
            "Resource": TOPIC_ARN,
        },
    ],
}
iam.put_role_policy(
    RoleName=DETECTOR_ROLE,
    PolicyName=DETECTOR_INLINE_POLICY,
    PolicyDocument=json.dumps(inline_policy),
)
print("Inline policy attached.")

# Lambda handler. Plain string so we can package it into a zip below.
# The handler wraps the stored anomaly SQL as a subquery and filters to the
# target_date passed in the event (or yesterday's UTC date when EventBridge
# fires the daily schedule with no payload). This makes the same SQL useful
# for a daily fire AND for replay testing against any historical date.
HANDLER_CODE = '''
import datetime as _dt
import json
import os
import time

import boto3

BUCKET = os.environ["BUCKET"]
DB = os.environ["DB"]
WG = os.environ["WG"]
TOPIC_ARN = os.environ["TOPIC_ARN"]
QUERY_KEY = os.environ["QUERY_KEY"]

athena = boto3.client("athena")
s3 = boto3.client("s3")
sns = boto3.client("sns")


def _wrap_with_target_date(sql, target_date):
    # Strip a trailing ORDER BY clause from the inner query because Trino
    # disallows ORDER BY inside an unsupported subquery position. Then wrap
    # the cleaned inner query and filter to the target_date.
    inner = sql.strip()
    upper = inner.upper()
    idx = upper.rfind("ORDER BY")
    if idx != -1 and "WHERE" not in upper[idx:]:
        inner = inner[:idx].rstrip()
    return (
        "SELECT * FROM (\\n" + inner + "\\n) anomalies WHERE usage_date = \\'" + target_date + "\\'"
    )


def handler(event, context):
    sql_obj = s3.get_object(Bucket=BUCKET, Key=QUERY_KEY)
    sql = sql_obj["Body"].read().decode("utf-8")

    target_date = (event or {}).get("target_date")
    if not target_date:
        target_date = (_dt.datetime.utcnow().date() - _dt.timedelta(days=1)).isoformat()

    final_sql = _wrap_with_target_date(sql, target_date)

    qid = athena.start_query_execution(
        QueryString=final_sql,
        QueryExecutionContext={"Database": DB},
        WorkGroup=WG,
    )["QueryExecutionId"]

    deadline = time.time() + 60
    while True:
        info = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = info["Status"]["State"]
        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            reason = info["Status"].get("StateChangeReason", "")
            return {"statusCode": 500, "state": state, "reason": reason, "target_date": target_date}
        if time.time() > deadline:
            return {"statusCode": 504, "state": state, "target_date": target_date}
        time.sleep(1.0)

    rows = []
    paginator = athena.get_paginator("get_query_results")
    for page in paginator.paginate(QueryExecutionId=qid):
        for row in page["ResultSet"]["Rows"]:
            rows.append([col.get("VarCharValue", "") for col in row["Data"]])

    # First row is the header; anomalies are anything beyond it.
    data_rows = rows[1:] if rows else []
    if not data_rows:
        return {"statusCode": 200, "anomalies": 0, "published": False, "target_date": target_date}

    body_lines = ["Cost anomaly detected for " + target_date + ":"]
    for r in data_rows:
        body_lines.append("  " + " | ".join(r))
    body = "\\n".join(body_lines)
    sns.publish(
        TopicArn=TOPIC_ARN,
        Subject="labrun cost-anomaly-detector: anomalies detected",
        Message=body,
    )
    return {"statusCode": 200, "anomalies": len(data_rows), "published": True, "target_date": target_date}
'''

# Build the zip in memory.
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("handler.py", HANDLER_CODE)
zip_bytes = zip_buf.getvalue()
print(f"Lambda zip built in memory ({len(zip_bytes)} bytes).")

# IAM role needs a moment to propagate before Lambda can assume it.
print("Lambda is warming up its container... letting the role propagate...")
time.sleep(15)

# Create or update the function.
role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{DETECTOR_ROLE}"
env = {
    "BUCKET": BUCKET_NAME,
    "DB": DB,
    "TABLE": TABLE,
    "WG": WG_NAME,
    "TOPIC_ARN": TOPIC_ARN,
    "QUERY_KEY": ANOMALY_QUERY_KEY,
}
try:
    # YOUR CODE: Create the Lambda using lam.create_function() with
    # FunctionName=DETECTOR_NAME, Runtime="python3.13", Role=role_arn,
    # Handler="handler.handler", Code={"ZipFile": zip_bytes}, Timeout=120,
    # Environment={"Variables": env}, Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    # MemorySize=256, Publish=True.
    print(f"Created Lambda function: {DETECTOR_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceConflictException":
        # Already exists. Update code and config.
        lam.update_function_code(FunctionName=DETECTOR_NAME, ZipFile=zip_bytes, Publish=True)
        lam.get_waiter("function_updated").wait(FunctionName=DETECTOR_NAME)
        lam.update_function_configuration(
            FunctionName=DETECTOR_NAME,
            Role=role_arn,
            Handler="handler.handler",
            Environment={"Variables": env},
            Timeout=120,
        )
        lam.get_waiter("function_updated").wait(FunctionName=DETECTOR_NAME)
        print(f"Lambda function {DETECTOR_NAME} already existed; updated code and config.")
    else:
        raise

# Wait for the function to be ready (state Active).
lam.get_waiter("function_active").wait(FunctionName=DETECTOR_NAME)
print("Lambda is Active.")

# First invoke: anomaly day. Detector should publish to SNS.
print("First invoke (anomaly date). The detector should find one row and publish.")
invoke_anomaly = lam.invoke(
    FunctionName=DETECTOR_NAME,
    InvocationType="RequestResponse",
    Payload=json.dumps({"target_date": ANOMALY_DATE.isoformat()}).encode("utf-8"),
)
payload_anomaly = invoke_anomaly["Payload"].read().decode("utf-8")
print(f"Anomaly-day invoke result: {payload_anomaly}")

print("Second invoke (clean date). The detector should find zero rows and stay quiet.")
invoke_clean = lam.invoke(
    FunctionName=DETECTOR_NAME,
    InvocationType="RequestResponse",
    Payload=json.dumps({"target_date": CLEAN_DATE.isoformat()}).encode("utf-8"),
)
payload_clean = invoke_clean["Payload"].read().decode("utf-8")
print(f"Clean-day invoke result: {payload_clean}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Lambda detector publishes to SNS on the anomaly date and stays
# silent on a clean replay. Verified via CloudWatch NumberOfMessagesPublished
# for the topic, with a 60-second metric publish lag buffer.

def checkpoint_3(session):
    try:
        lam_client = boto3.client(
            "lambda",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        cw = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            lam_client.get_function(FunctionName=DETECTOR_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Lambda function {DETECTOR_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        def metric_sum(start, end):
            r = cw.get_metric_statistics(
                Namespace="AWS/SNS",
                MetricName="NumberOfMessagesPublished",
                Dimensions=[{"Name": "TopicName", "Value": TOPIC_NAME}],
                StartTime=start,
                EndTime=end,
                Period=60,
                Statistics=["Sum"],
            )
            return sum(p.get("Sum", 0) for p in r.get("Datapoints", []))

        # Baseline window (last 5 minutes).
        now = datetime.now(timezone.utc)
        baseline = metric_sum(now - timedelta(minutes=5), now)
        print(f"Baseline SNS publishes in last 5 minutes (pre-anomaly invoke): {baseline}")

        # Invoke with anomaly date.
        resp = lam_client.invoke(
            FunctionName=DETECTOR_NAME,
            InvocationType="RequestResponse",
            Payload=json.dumps({"target_date": ANOMALY_DATE.isoformat()}).encode("utf-8"),
        )
        anomaly_payload = resp["Payload"].read().decode("utf-8")
        if resp.get("FunctionError"):
            return CheckpointResult(
                status="fail",
                error_reason=f"Lambda raised on anomaly-date invoke: {anomaly_payload}",
            )
        try:
            anomaly_result = json.loads(anomaly_payload)
        except Exception:
            return CheckpointResult(
                status="fail",
                error_reason=f"Lambda did not return JSON on anomaly invoke: {anomaly_payload!r}",
            )
        if anomaly_result.get("published") is not True:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Lambda did not publish to SNS on the anomaly invoke. Returned: {anomaly_result}. "
                    f"Check that the handler reads the SQL from S3 and calls sns.publish when rows return."
                ),
            )

        # CloudWatch metric publish lag is real. Sleeping 60 seconds before re-reading.
        print("CloudWatch metric publish lag is real. Sleeping 60 seconds before re-reading...")
        time.sleep(60)
        post_anomaly = metric_sum(now - timedelta(minutes=5), datetime.now(timezone.utc))
        delta_anomaly = post_anomaly - baseline
        print(f"SNS publishes after anomaly invoke: {post_anomaly} (delta: {delta_anomaly})")
        if delta_anomaly < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CloudWatch did not record any new SNS publish for topic {TOPIC_NAME} "
                    f"after the anomaly invoke. Verify the handler's sns.publish call and the "
                    f"role's sns:Publish permission."
                ),
            )

        # Invoke with clean date. No new publishes expected.
        baseline_clean = post_anomaly
        resp = lam_client.invoke(
            FunctionName=DETECTOR_NAME,
            InvocationType="RequestResponse",
            Payload=json.dumps({"target_date": CLEAN_DATE.isoformat()}).encode("utf-8"),
        )
        clean_payload = resp["Payload"].read().decode("utf-8")
        if resp.get("FunctionError"):
            return CheckpointResult(
                status="fail",
                error_reason=f"Lambda raised on clean-date invoke: {clean_payload}",
            )
        try:
            clean_result = json.loads(clean_payload)
        except Exception:
            return CheckpointResult(
                status="fail",
                error_reason=f"Lambda did not return JSON on clean invoke: {clean_payload!r}",
            )
        if clean_result.get("published") is True:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Lambda published to SNS on the clean replay invoke. Returned: {clean_result}. "
                    f"The handler must only publish when the Athena result has rows beyond the header."
                ),
            )

        print("Asking SNS if the publish actually landed... clean replay did not move the meter, as expected.")
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The detector should be silent on clean days. Read the row count from the Athena result, skip the header row, and publish only when there is at least one anomaly row. If your handler is publishing on every invoke, you forgot the `if not data_rows: return` early-exit.

</details>

<details><summary>Hint 2 (direction)</summary>

The handler shape is: `s3.get_object` the SQL text, wrap it in a CTE that filters to the `target_date` from the event (or yesterday's UTC date when EventBridge fires with no payload), `athena.start_query_execution` with `QueryString=final_sql` and `WorkGroup=WG`, poll `athena.get_query_execution` until `State=SUCCEEDED`, paginate `athena.get_query_results`, and publish to SNS only when `len(rows) > 1` after counting the header. The Lambda role's inline policy must grant `athena:StartQueryExecution`, `athena:GetQueryExecution`, `athena:GetQueryResults`, `s3:GetObject`, `s3:PutObject` (Athena writes results), `s3:ListBucket`, `glue:GetTable`, and `sns:Publish`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3. The handler code is embedded as a string and packaged into a zip in memory; you do not need to mess with the file system. Replace the YOUR CODE blocks with this:

```python
# SNS topic
topic_resp = sns.create_topic(
    Name=TOPIC_NAME,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
topic_arn = topic_resp["TopicArn"]

# Lambda create_function
lam.create_function(
    FunctionName=DETECTOR_NAME,
    Runtime="python3.13",
    Role=role_arn,
    Handler="handler.handler",
    Code={"ZipFile": zip_bytes},
    Timeout=120,
    MemorySize=256,
    Environment={"Variables": env},
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    Publish=True,
)
```

The handler body inside `HANDLER_CODE` (already filled in for you) is the canonical shape: read SQL from S3, wrap the stored anomaly query as a CTE and filter `WHERE usage_date = <target_date>`, start the Athena query, poll until SUCCEEDED, paginate `get_query_results`, count rows beyond the header, publish only when `len(data_rows) > 0`. The `target_date` event field is what makes the same Lambda useful for both the daily fire (no payload, defaults to yesterday UTC) and direct invokes for replay testing.

</details>

**Wallet check.** Two Lambda invokes (free at lab volume; the first 1M requests/month and 400k GB-s are always free). Two Athena queries scanning ~5 KB each (still microcents). Two SNS publishes (free at lab volume; 1M publishes/month included). Running cost: still about two cents. The detector itself is free; the entire bill so far is the Athena scan rounding.

## Task 4: Schedule the detector to run daily via EventBridge

The Lambda invoked from the notebook is a manual fire. Production wants the detector to run once a day on its own, paging the on-call without anyone watching it. EventBridge's schedule expressions are how AWS does cron in the cloud; the cert tests this pattern directly.

Build it in this order:
1. Create the EventBridge rule `labrun-cost-anomaly-detection-daily` on the default bus with `ScheduleExpression="cron(15 8 * * ? *)"`. That cron fires at 08:15 UTC every day. The `?` is the day-of-month placeholder when day-of-week is unspecified; EventBridge requires it.
2. Add the detector Lambda as Target Id `"1"` with the function ARN. The cleanup adapter expects target Id `"1"`, so this matters.
3. Add a Lambda resource policy permission so `events.amazonaws.com` can invoke the function on behalf of the rule. Without this, the EventBridge fire fails silently at runtime with an AccessDenied buried in CloudTrail.

EventBridge cron expressions are notoriously picky. The checkpoint reads the rule with `events.describe_rule` and asserts the schedule expression and target ARN match exactly.

In [ ]:
# NBVAL_SKIP
# Task 4: EventBridge daily schedule, Lambda target, and Lambda invoke
# permission for events.amazonaws.com.

events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lam = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

SCHEDULE_EXPRESSION = "cron(15 8 * * ? *)"

# YOUR CODE: Create the EventBridge rule using events.put_rule() with
# Name=SCHEDULE_NAME, ScheduleExpression=SCHEDULE_EXPRESSION, State="ENABLED",
# Description="labrun cost anomaly detector daily fire", and
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Save the response to rule_resp; the rule ARN is rule_resp["RuleArn"].
rule_arn = rule_resp["RuleArn"]
print(f"EventBridge rule created: {SCHEDULE_NAME}")
print(f"Schedule expression: {SCHEDULE_EXPRESSION}")
print(f"Rule ARN: {rule_arn}")

# Add the Lambda as a target. Target Id must be "1" so cleanup can find it.
# YOUR CODE: Call events.put_targets() with Rule=SCHEDULE_NAME and
# Targets=[{"Id": "1", "Arn": DETECTOR_LAMBDA_ARN}]. EventBridge target
# payload is JSON; you can leave Input default for a daily fire that uses the
# Lambda's environment for everything it needs.
print(f"Target added: {DETECTOR_LAMBDA_ARN}")

# Grant EventBridge permission to invoke the Lambda. Without this the daily
# fire silently fails with AccessDenied buried in CloudTrail.
try:
    lam.add_permission(
        FunctionName=DETECTOR_NAME,
        StatementId=f"AllowEventBridgeInvoke-{SCHEDULE_NAME}",
        Action="lambda:InvokeFunction",
        Principal="events.amazonaws.com",
        SourceArn=rule_arn,
    )
    print("Lambda invoke permission granted to events.amazonaws.com.")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceConflictException":
        print("Lambda invoke permission already exists, reusing.")
    else:
        raise

print()
print("EventBridge cron expressions are notoriously picky. Validating...")
time.sleep(2)
describe = events.describe_rule(Name=SCHEDULE_NAME)
print(f"  Name: {describe['Name']}")
print(f"  ScheduleExpression: {describe['ScheduleExpression']}")
print(f"  State: {describe['State']}")
targets = events.list_targets_by_rule(Rule=SCHEDULE_NAME)["Targets"]
for t in targets:
    print(f"  Target Id={t['Id']!r} Arn={t['Arn']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: EventBridge rule has a specific-hour daily cron and the detector
# Lambda as a target.

def checkpoint_4(session):
    try:
        events_client = boto3.client(
            "events",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            rule = events_client.describe_rule(Name=SCHEDULE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"EventBridge rule {SCHEDULE_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        sched = rule.get("ScheduleExpression", "")
        # Accept any specific-hour daily cron of the form cron(M H * * ? *)
        # where M and H are integer minutes 0-59 / hours 0-23.
        ok = False
        if sched.startswith("cron(") and sched.endswith(")"):
            parts = sched[5:-1].split()
            if len(parts) == 6:
                minute, hour, dom, month, dow, year = parts
                try:
                    m_int = int(minute)
                    h_int = int(hour)
                    if (0 <= m_int <= 59 and 0 <= h_int <= 23
                            and dom == "*" and month == "*"
                            and dow == "?" and year == "*"):
                        ok = True
                except ValueError:
                    ok = False
        if not ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Rule {SCHEDULE_NAME} schedule is {sched!r}; expected a specific-hour daily "
                    f"cron of shape cron(<minute> <hour> * * ? *), e.g. cron(15 8 * * ? *)."
                ),
            )
        if rule.get("State") != "ENABLED":
            return CheckpointResult(
                status="fail",
                error_reason=f"Rule {SCHEDULE_NAME} state is {rule.get('State')!r}; expected 'ENABLED'.",
            )

        targets = events_client.list_targets_by_rule(Rule=SCHEDULE_NAME).get("Targets", [])
        target_arns = [t.get("Arn", "") for t in targets]
        if DETECTOR_LAMBDA_ARN not in target_arns:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Rule {SCHEDULE_NAME} targets do not include {DETECTOR_LAMBDA_ARN!r}. "
                    f"Found: {target_arns}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

EventBridge schedule expressions are cron-style but they require six fields, not the five Unix cron uses, and the day-of-month plus day-of-week slots conflict (you have to put `?` in one when you set the other). The default bus's rule does not need a custom event bus name; omit `EventBusName` and AWS routes to default.

</details>

<details><summary>Hint 2 (direction)</summary>

`cron(15 8 * * ? *)` fires at 08:15 UTC every day. Add the Lambda as Target Id `"1"` so the cleanup adapter can find it (`events.remove_targets(Rule=..., Ids=["1"], Force=True)` is what the AWS cleanup adapter calls). After `put_targets`, grant the rule permission to invoke the Lambda via `lam.add_permission(StatementId=..., Action="lambda:InvokeFunction", Principal="events.amazonaws.com", SourceArn=rule_arn)` or the daily fire will silently AccessDenied.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
rule_resp = events.put_rule(
    Name=SCHEDULE_NAME,
    ScheduleExpression="cron(15 8 * * ? *)",
    State="ENABLED",
    Description="labrun cost anomaly detector daily fire",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
rule_arn = rule_resp["RuleArn"]

events.put_targets(
    Rule=SCHEDULE_NAME,
    Targets=[{"Id": "1", "Arn": DETECTOR_LAMBDA_ARN}],
)

lam.add_permission(
    FunctionName=DETECTOR_NAME,
    StatementId=f"AllowEventBridgeInvoke-{SCHEDULE_NAME}",
    Action="lambda:InvokeFunction",
    Principal="events.amazonaws.com",
    SourceArn=rule_arn,
)
```

</details>

**Wallet check.** EventBridge rules on the default bus are free for AWS service events. `put_rule`, `put_targets`, and the Lambda resource policy `add_permission` are all free operations. Running cost: about three cents total for the session (Athena scans), give or take the Lambda invoke cost which still rounds to zero at this volume.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. No critical (hourly-billed) resources in this lab; the canonical
# summary always reports zero critical destructions.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $0.20 to $0.80** depending on debug re-runs of the anomaly query against the seeded CUR. Athena scans dominated the meter at $5 / TB on roughly 100 MB total over the session (under 10 query executions, each scanning a few hundred KB to a couple MB). Lambda, SNS, EventBridge, Glue catalog, S3 storage, and S3 requests all stayed inside their always-free tiers. The cleanup cell above deleted the EventBridge rule, the Lambda function, the SNS topic, the IAM role, the Athena workgroup, the Glue table and database, and the S3 bucket so nothing keeps accruing. Check your AWS Billing console in 24 hours to confirm.

## These are not graded. They are for you.

Three questions worth sitting with before you ship the next version of this detector to production.

1. This lab seeded its own CUR-shaped table because real CUR delivery is daily and lags up to 24 hours. If you switched on real CUR for your account tomorrow, three things change in the detector's runtime behavior: the table schema grows from three columns to roughly thirty (line item type, savings plan attribution, tags, reservation amortization, blended vs unblended, etc.); the latest available `usage_date` is always yesterday or earlier (the detector's daily fire at 08:15 UTC reads data that landed sometime in the prior six hours); and the row volume per day jumps from five rows (one per service) to thousands (one per resource per usage type per pricing dimension). Walk through how each of those changes the SQL you wrote in Task 2. The 3-sigma logic stays; the GROUP BY changes.

2. 3-sigma over a 14-day rolling window is one specific detector choice. Identify two production scenarios where 3-sigma would generate too many false positives or miss real anomalies. Hint: a service whose cost follows a bimodal pattern (low weekends, high weekdays) breaks the unimodal-Gaussian assumption that 3-sigma sits on top of; a slow drift upward over weeks (a leaked log group eating CloudWatch storage at $0.03/GB) never crosses the 3-sigma envelope because the mean creeps up with it.

3. The detector publishes to SNS because SNS fans out to email, Slack, PagerDuty, and HTTP endpoints without a code change in the Lambda. Walk through one production reason the SNS topic should NOT be the only paging target. Hint: SNS has an at-most-once delivery contract for HTTPS subscribers; a single dropped retry to the on-call's webhook means no page. The defensive pattern is publishing to SNS AND writing the anomaly to a durable store (S3, DynamoDB) so the on-call has a forensic trail when the page does not land.

## Exam-Style Practice

**Q1.** A data engineer wires a Lambda detector to read AWS Cost and Usage Report data daily via Athena and publish to SNS when anomalies are detected. The Lambda runs from an EventBridge schedule at 08:15 UTC. Most days are quiet, but on 1 day in 30 a real cost spike fires the alert. The on-call team reports they sometimes get duplicate alerts a few minutes apart for the same anomaly day. What is the most likely cause?

A) Athena charged twice for the same query because the BytesScannedCutoffPerQuery cap was set too low and the query retried.

B) The EventBridge rule has two targets pointing at the same Lambda by accident.

C) The Lambda timed out and EventBridge retried automatically; the second invoke published a duplicate page before the first invoke's SNS publish finished.

D) SNS uses at-least-once delivery for the HTTP subscriber, and the on-call's webhook is generating an alert per delivery attempt.

<details><summary>Show answer</summary>

**C**.

- A) Wrong. `BytesScannedCutoffPerQuery` is a per-query data-scan limit; if a query exceeds it, Athena fails the query and Lambda surfaces that as a non-200 return. There is no automatic retry from Athena, and a failed query does not publish to SNS.
- B) Wrong. The cleanup adapter expects Target Id `"1"`; if two targets were registered the second one would be Id `"2"` and the rule would still fan to two targets, but EventBridge does not silently dedupe target lists. This scenario is rare and would have been caught at deploy.
- C) Right. EventBridge scheduled rules retry on Lambda invoke failures (including timeouts) up to twice within 6 hours by default. A long-running Athena query can push the Lambda past its 60- or 120-second timeout; the retry then invokes the function again, the second execution completes, both publish to SNS, on-call gets two pages. The fix is increasing the Lambda timeout, narrowing the query, or making the publish idempotent via an external dedupe table keyed by anomaly day.
- D) Wrong. SNS HTTPS subscribers receive at-least-once delivery from SNS itself, but SNS retries against a 5xx-returning endpoint, not on a 200; if the webhook returned 200, SNS does not retry. This is the wrong layer of the stack to blame.

</details>

**Q2.** A data engineer needs to detect cost anomalies in an AWS account where one service (CloudWatch Logs) follows a bimodal distribution: low on weekends, high on weekdays. The team initially deployed a 3-sigma detector over a 14-day rolling window and the on-call gets paged every Monday morning. What is the better detector design?

A) Stratify the rolling baseline by day-of-week (compute mean and stddev separately for each weekday) so weekday spikes are scored against the weekday baseline, not the mixed weekly mean.

B) Drop the 3-sigma threshold to 5-sigma so only larger spikes fire.

C) Switch to an isolation forest over the same rolling window.

D) Move the detector to fire weekly instead of daily so the Monday spike is averaged out.

<details><summary>Show answer</summary>

**A**.

- A) Right. The Monday false positives come from the bimodal distribution: the 14-day mean blends weekend lows and weekday highs, so Monday's normal cost looks like a 3-sigma spike against the blended baseline. Stratifying by day-of-week (or using a seasonality-aware model like Prophet or STL decomposition for production) is the canonical fix when the cost time series has known periodicity.
- B) Wrong. Moving from 3-sigma to 5-sigma reduces sensitivity globally; you trade one set of false positives for missed real anomalies, which is the worse trade for cost monitoring where the goal is catching $400 spikes inside 24 hours.
- C) Wrong. Isolation forests work well for high-dimensional anomaly detection but are overkill for a univariate cost-per-service time series and they do not naturally encode the day-of-week structure. The lift over a stratified Gaussian is not worth the operational complexity for this problem.
- D) Wrong. A weekly fire defeats the purpose of catching anomalies before finance does. The Monday spike still pages the on-call; you just batch the pages.

</details>

**Q3.** A data engineer's cost-anomaly detector reads the Cost and Usage Report via Athena and publishes to SNS on anomalies. The Lambda role's inline policy grants `athena:StartQueryExecution`, `athena:GetQueryExecution`, `athena:GetQueryResults`, `s3:GetObject` on the bucket, `sns:Publish` on the topic, and `glue:GetTable` on the table. The Lambda fires successfully but every Athena query fails with `Access denied to S3 location: s3://bucket/athena-results/`. What is missing from the role's permissions?

A) `s3:PutObject` on the bucket so Athena can write the query results file to the workgroup's results location.

B) `athena:GetWorkGroup` so the role can read the workgroup configuration.

C) `lakeformation:GetDataAccess` so Lake Formation grants the read.

D) `kms:Decrypt` on the bucket's KMS key.

<details><summary>Show answer</summary>

**A**.

- A) Right. Athena writes the query result CSV to the workgroup's `OutputLocation` before `GetQueryResults` can return. Without `s3:PutObject` on the results prefix (and `s3:GetBucketLocation` so Athena can locate the bucket region) the query fails at write time with the AccessDenied surface message naming the results path. This is the single most common missing permission in Athena-via-Lambda setups.
- B) Wrong. `athena:GetWorkGroup` is needed only if the Lambda reads the workgroup config explicitly; specifying the workgroup name on `start_query_execution` does not require it. The error message would not mention an S3 path if this were the cause.
- C) Wrong. Lake Formation is opt-in and not present unless the account has registered the bucket with LF; the error message would surface a Lake Formation denial, not an S3 AccessDenied.
- D) Wrong. KMS would surface as a KMS denial, not an S3 AccessDenied. The results bucket in this lab is SSE-S3, not SSE-KMS, so there is no key to decrypt.

</details>